# Phase 1 – Audio Standardisation
**Gujarati Accent Classifier**

Objective: convert every raw recording (M4A / OGG) into one common format so all later phases (quality analysis, MFCCs, modelling) see identical input:

| Property | Specification |
|---|---|
| Format | WAV |
| Encoding | PCM signed 16-bit |
| Sample rate | 16,000 Hz |
| Channels | Mono |

Only the *format* changes: no noise reduction, pitch/speed change, silence removal, filtering or augmentation. Raw files are never modified. Speakers were allowed to adapt the prompts to their regional vocabulary, so transcripts are intentionally **not** created or normalised.

**Surti** recordings are pending collection; they will be picked up automatically when added to `dataset/south-Gujarat/surti/` and the notebook is re-run.

Reusable logic lives in `src/audio_utils.py`; settings in `src/config.py`.

## 2. Import dependencies

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Locate the project root (the folder containing `dataset/`) so no absolute paths are needed.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dataset").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import config as cfg
import audio_utils as au

pd.set_option("display.width", 200, "display.max_columns", 20, "display.max_colwidth", 70)
print("FFmpeg:", au.find_ffmpeg())

FFmpeg: C:\Users\91966\AppData\Local\Programs\Python\Python312\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v9.0.2.exe


## 3. Project and dataset paths
All paths derive from the project root.

In [2]:
print("Project root :", PROJECT_ROOT)
print("Raw dataset  :", cfg.DATASET_DIR.relative_to(PROJECT_ROOT))
print("Processed WAV:", cfg.PROCESSED_WAV_DIR.relative_to(PROJECT_ROOT))
print("Metadata CSV :", cfg.METADATA_CSV.relative_to(PROJECT_ROOT))

Project root : D:\university\year4\NLP\nlp-researchPaper
Raw dataset  : dataset
Processed WAV: dataset\processed\wav
Metadata CSV : dataset\recordings_metadata.csv


## 4. Accent and speaker mapping
Defined in `src/config.py` (`ACCENTS`). An accent is identified by the **actual raw folder** a file sits in (e.g. `kutch/` → Kutchi). One speaker per accent currently, so all 7 recordings of an accent share one speaker ID. To add or change accents, edit the config only.

In [3]:
pd.DataFrame([{"accent": a.accent, "region": a.region, "speaker_id": a.speaker_id,
               "folder_names matched": ", ".join(a.folder_names)} for a in cfg.ACCENTS])

,accent,region,speaker_id,folder_names matched
0,Ahmedabad,Central Gujarat,AHM_S01,ahmedabad
1,Kutchi,Kutch,KUT_S01,"kutch, kutchi"
2,Mehsani,North Gujarat,MEH_S01,mehsani
3,Kathiyawadi,Saurashtra,KAT_S01,kathiyawadi
4,Surti,South Gujarat,SUR_S01,surti


## 5. Discover original audio files
Recursive scan of `dataset/`, supported extensions only (`.m4a`, `.ogg`), the `processed/` folder excluded. Files in a folder that matches no configured accent are listed separately rather than silently dropped or guessed.

In [4]:
recordings, unmapped = au.discover_recordings()
print(f"Discovered {len(recordings)} recordings mapped to an accent; {len(unmapped)} in unmapped folders.")
pd.DataFrame([{"recording_id": r.recording_id, "speaker_id": r.speaker_id, "accent": r.accent,
               "format": r.original_format, "original": r.source_path.relative_to(PROJECT_ROOT).as_posix(),
               "will_write": r.processed_path.relative_to(PROJECT_ROOT).as_posix()} for r in recordings])

Discovered 28 recordings mapped to an accent; 0 in unmapped folders.


,recording_id,speaker_id,accent,format,original,will_write
0,AHM_S01_R01,AHM_S01,Ahmedabad,M4A,dataset/central-Gujarat/ahmedabad/Ahmedabad-01-GAC.m4a,dataset/processed/wav/Ahmedabad/Ahmedabad-01-GAC.wav
1,AHM_S01_R02,AHM_S01,Ahmedabad,M4A,dataset/central-Gujarat/ahmedabad/Ahmedabad-02-GAC.m4a,dataset/processed/wav/Ahmedabad/Ahmedabad-02-GAC.wav
2,AHM_S01_R03,AHM_S01,Ahmedabad,M4A,dataset/central-Gujarat/ahmedabad/Ahmedabad-03-GAC.m4a,dataset/processed/wav/Ahmedabad/Ahmedabad-03-GAC.wav
3,AHM_S01_R04,AHM_S01,Ahmedabad,M4A,dataset/central-Gujarat/ahmedabad/Ahmedabad-04-GAC.m4a,dataset/processed/wav/Ahmedabad/Ahmedabad-04-GAC.wav
4,AHM_S01_R05,AHM_S01,Ahmedabad,M4A,dataset/central-Gujarat/ahmedabad/Ahmedabad-05-GAC.m4a,dataset/processed/wav/Ahmedabad/Ahmedabad-05-GAC.wav
5,AHM_S01_R06,AHM_S01,Ahmedabad,M4A,dataset/central-Gujarat/ahmedabad/Ahmedabad-06-GAC.m4a,dataset/processed/wav/Ahmedabad/Ahmedabad-06-GAC.wav
6,AHM_S01_R07,AHM_S01,Ahmedabad,M4A,dataset/central-Gujarat/ahmedabad/Ahmedabad-07-GAC.m4a,dataset/processed/wav/Ahmedabad/Ahmedabad-07-GAC.wav
7,KUT_S01_R01,KUT_S01,Kutchi,M4A,dataset/kutch/Kutchi-01 -GAC.m4a,dataset/processed/wav/Kutchi/Kutchi-01_-GAC.wav
8,KUT_S01_R02,KUT_S01,Kutchi,M4A,dataset/kutch/Kutchi-02-GAC.m4a,dataset/processed/wav/Kutchi/Kutchi-02-GAC.wav
9,KUT_S01_R03,KUT_S01,Kutchi,M4A,dataset/kutch/Kutchi-03-GAC.m4a,dataset/processed/wav/Kutchi/Kutchi-03-GAC.wav


## 6. Inspect original audio properties
SoundFile reads OGG directly; for M4A (unsupported by libsndfile) properties come from FFmpeg's stream info.

In [5]:
orig = pd.DataFrame([{"recording_id": r.recording_id, "format": r.original_format,
                      **au.inspect_original(r.source_path)} for r in recordings])
display(orig)
print(orig.groupby("format")[["sample_rate", "channels"]].agg(lambda s: sorted(s.astype(str).unique())))

,recording_id,format,sample_rate,channels,duration_seconds,reader
0,AHM_S01_R01,M4A,48000,1,14.190,ffmpeg
1,AHM_S01_R02,M4A,48000,1,11.970,ffmpeg
2,AHM_S01_R03,M4A,48000,1,11.990,ffmpeg
3,AHM_S01_R04,M4A,48000,1,12.160,ffmpeg
4,AHM_S01_R05,M4A,48000,1,17.260,ffmpeg
5,AHM_S01_R06,M4A,48000,1,12.120,ffmpeg
6,AHM_S01_R07,M4A,48000,1,16.450,ffmpeg
7,KUT_S01_R01,M4A,48000,1,15.150,ffmpeg
8,KUT_S01_R02,M4A,48000,1,16.960,ffmpeg
9,KUT_S01_R03,M4A,48000,1,15.400,ffmpeg


           sample_rate channels
format                         
M4A            [48000]      [1]
OGG     [16000, 48000]      [1]


## 7. Convert recordings with FFmpeg
`-ac 1 -ar 16000 -c:a pcm_s16le` is the entire transformation. Output goes to `dataset/processed/wav/<Accent>/`, keeping the original file name (whitespace → `_`). A failure on one file is recorded and the loop continues. Re-running skips outputs that already exist, are valid and are newer than their source; pass `force=True` to reconvert.

In [6]:
results = au.process_recordings(recordings, force=False)
pd.DataFrame([{"recording_id": r.recording.recording_id, "status": r.conversion_status,
               "message": r.message} for r in results])

,recording_id,status,message
0,AHM_S01_R01,already_converted,valid output exists; skipped
1,AHM_S01_R02,already_converted,valid output exists; skipped
2,AHM_S01_R03,already_converted,valid output exists; skipped
3,AHM_S01_R04,already_converted,valid output exists; skipped
4,AHM_S01_R05,already_converted,valid output exists; skipped
5,AHM_S01_R06,already_converted,valid output exists; skipped
6,AHM_S01_R07,already_converted,valid output exists; skipped
7,KUT_S01_R01,already_converted,valid output exists; skipped
8,KUT_S01_R02,already_converted,valid output exists; skipped
9,KUT_S01_R03,already_converted,valid output exists; skipped


## 8. Validate converted WAV files
Every processed file is re-opened independently with SoundFile and checked for: existence, readability (full decode), 16,000 Hz, 1 channel, `PCM_16`, duration > 0, and frame count consistent with the header.

In [7]:
validation = pd.DataFrame([{"recording_id": r.recording.recording_id, "accent": r.recording.accent,
                            **{k: r.validation.get(k) for k in ("exists", "readable", "sample_rate", "channels",
                                                                 "audio_subtype", "duration_seconds")},
                            "valid": r.valid, "problems": "; ".join(r.validation.get("problems", []))}
                           for r in results])
validation

,recording_id,accent,exists,readable,sample_rate,channels,audio_subtype,duration_seconds,valid,problems
0,AHM_S01_R01,Ahmedabad,True,True,16000,1,PCM_16,14.187,True,
1,AHM_S01_R02,Ahmedabad,True,True,16000,1,PCM_16,11.968,True,
2,AHM_S01_R03,Ahmedabad,True,True,16000,1,PCM_16,11.989,True,
3,AHM_S01_R04,Ahmedabad,True,True,16000,1,PCM_16,12.160,True,
4,AHM_S01_R05,Ahmedabad,True,True,16000,1,PCM_16,17.259,True,
5,AHM_S01_R06,Ahmedabad,True,True,16000,1,PCM_16,12.117,True,
6,AHM_S01_R07,Ahmedabad,True,True,16000,1,PCM_16,16.448,True,
7,KUT_S01_R01,Kutchi,True,True,16000,1,PCM_16,15.147,True,
8,KUT_S01_R02,Kutchi,True,True,16000,1,PCM_16,16.960,True,
9,KUT_S01_R03,Kutchi,True,True,16000,1,PCM_16,15.403,True,


## 9. Generate metadata CSV
Written to `dataset/recordings_metadata.csv`. `recording_type` is left empty (not known); no transcripts are invented. Paths are relative to the project root.

In [8]:
metadata = au.build_metadata(results)
csv_path = au.save_metadata(metadata)
print("Saved:", csv_path.relative_to(PROJECT_ROOT))
metadata

Saved: dataset\recordings_metadata.csv


,recording_id,speaker_id,accent,region,recording_type,original_file_path,processed_file_path,original_format,duration_seconds,sample_rate,channels,audio_subtype,conversion_status
0,AHM_S01_R01,AHM_S01,Ahmedabad,Central Gujarat,,dataset/central-Gujarat/ahmedabad/Ahmedabad-01-GAC.m4a,dataset/processed/wav/Ahmedabad/Ahmedabad-01-GAC.wav,M4A,14.187,16000,1,PCM_16,success
1,AHM_S01_R02,AHM_S01,Ahmedabad,Central Gujarat,,dataset/central-Gujarat/ahmedabad/Ahmedabad-02-GAC.m4a,dataset/processed/wav/Ahmedabad/Ahmedabad-02-GAC.wav,M4A,11.968,16000,1,PCM_16,success
2,AHM_S01_R03,AHM_S01,Ahmedabad,Central Gujarat,,dataset/central-Gujarat/ahmedabad/Ahmedabad-03-GAC.m4a,dataset/processed/wav/Ahmedabad/Ahmedabad-03-GAC.wav,M4A,11.989,16000,1,PCM_16,success
3,AHM_S01_R04,AHM_S01,Ahmedabad,Central Gujarat,,dataset/central-Gujarat/ahmedabad/Ahmedabad-04-GAC.m4a,dataset/processed/wav/Ahmedabad/Ahmedabad-04-GAC.wav,M4A,12.160,16000,1,PCM_16,success
4,AHM_S01_R05,AHM_S01,Ahmedabad,Central Gujarat,,dataset/central-Gujarat/ahmedabad/Ahmedabad-05-GAC.m4a,dataset/processed/wav/Ahmedabad/Ahmedabad-05-GAC.wav,M4A,17.259,16000,1,PCM_16,success
5,AHM_S01_R06,AHM_S01,Ahmedabad,Central Gujarat,,dataset/central-Gujarat/ahmedabad/Ahmedabad-06-GAC.m4a,dataset/processed/wav/Ahmedabad/Ahmedabad-06-GAC.wav,M4A,12.117,16000,1,PCM_16,success
6,AHM_S01_R07,AHM_S01,Ahmedabad,Central Gujarat,,dataset/central-Gujarat/ahmedabad/Ahmedabad-07-GAC.m4a,dataset/processed/wav/Ahmedabad/Ahmedabad-07-GAC.wav,M4A,16.448,16000,1,PCM_16,success
7,KUT_S01_R01,KUT_S01,Kutchi,Kutch,,dataset/kutch/Kutchi-01 -GAC.m4a,dataset/processed/wav/Kutchi/Kutchi-01_-GAC.wav,M4A,15.147,16000,1,PCM_16,success
8,KUT_S01_R02,KUT_S01,Kutchi,Kutch,,dataset/kutch/Kutchi-02-GAC.m4a,dataset/processed/wav/Kutchi/Kutchi-02-GAC.wav,M4A,16.960,16000,1,PCM_16,success
9,KUT_S01_R03,KUT_S01,Kutchi,Kutch,,dataset/kutch/Kutchi-03-GAC.m4a,dataset/processed/wav/Kutchi/Kutchi-03-GAC.wav,M4A,15.403,16000,1,PCM_16,success


## 10. Accent-wise conversion summary
Every configured accent is listed, so Surti shows `0` until its recordings are collected.

In [9]:
summary = au.summarise(results, unmapped)
per_accent = pd.DataFrame({
    "accent": [a.accent for a in cfg.ACCENTS],
    "region": [a.region for a in cfg.ACCENTS],
    "speaker_id": [a.speaker_id for a in cfg.ACCENTS],
    "valid_recordings": [summary["per_accent_valid"][a.accent] for a in cfg.ACCENTS],
})
per_accent["note"] = per_accent["valid_recordings"].map(lambda n: "pending future collection" if n == 0 else "")
display(per_accent)
print("Total valid recordings:", per_accent["valid_recordings"].sum())

,accent,region,speaker_id,valid_recordings,note
0,Ahmedabad,Central Gujarat,AHM_S01,7,
1,Kutchi,Kutch,KUT_S01,7,
2,Mehsani,North Gujarat,MEH_S01,7,
3,Kathiyawadi,Saurashtra,KAT_S01,7,
4,Surti,South Gujarat,SUR_S01,0,pending future collection


Total valid recordings: 28


## 11. Errors and validation results

In [10]:
report = au.format_report(summary)
print(report)
(cfg.REPORTS_DIR).mkdir(parents=True, exist_ok=True)
(cfg.REPORTS_DIR / "validation_report.txt").write_text(report + chr(10), encoding="utf-8")

failures = validation[~validation["valid"]]
if failures.empty:
    print(chr(10) + "All converted recordings passed validation.")
else:
    display(failures)

=== Phase 1 validation report ===
Total files discovered      : 28
Successfully converted      : 28
Failed conversions          : 0 
Validation failures         : 0 
Files in unmapped folders   : 0 

Valid recordings per accent:
  Ahmedabad   : 7
  Kutchi      : 7
  Mehsani     : 7
  Kathiyawadi : 7
  Surti       : 0 (pending future collection)

Total validated: 28

All converted recordings passed validation.


## Adding Surti later
1. Put the Surti recordings (`.m4a` / `.ogg`) in `dataset/south-Gujarat/surti/`.
2. Re-run this notebook (or `python scripts/run_phase1.py`).

Existing outputs are skipped, Surti files get speaker `SUR_S01` and are written to `dataset/processed/wav/Surti/`, and the metadata CSV is regenerated with all five accents. Nothing else needs to change.